<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 36px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  <b>01. Expectation-Maximization Deep Dive</b>
</div>

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">📑 Table of Contents</span>

- **1. [The EM Algorithm: Motivation & Theory](#-part-i-the-em-algorithm)**
    - 1.1 [Latent Variables and Incomplete Data](#latent-variables)
    - 1.2 [ELBO and the EM Framework](#elbo-and-em)
- **2. [Setup & Imports](#-part-ii-setup--imports)**
- **3. [EM for Gaussian Mixtures from Scratch](#-part-iii-em-from-scratch)**
    - 3.1 [E-Step: Computing Responsibilities](#e-step)
    - 3.2 [M-Step: Updating Parameters](#m-step)
    - 3.3 [Full EM Algorithm](#full-em)
- **4. [Visualizing EM Convergence](#-part-iv-visualizing-convergence)**
- **5. [EM vs Gradient Descent](#-part-v-em-vs-gd)**
- **6. [Connection to Variational Inference](#-part-vi-connection-to-vi)**
- **7. [Key Takeaways](#-part-vii-key-takeaways)**

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">1. The EM Algorithm: Motivation & Theory</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">1.1. Latent Variables and Incomplete Data</span>

- The **EM algorithm** (Dempster, Laird & Rubin, 1977) is a general method for finding MLE/MAP estimates when data is "incomplete" (has latent variables).
- For GMMs, the latent variable $z_i$ indicates which component generated observation $x_i$.
- If we knew $z_i$, fitting would be trivial. EM alternates between:
  - **E-step**: Infer the latent variables given current parameters
  - **M-step**: Update parameters given inferred latent variables

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">1.2. ELBO and the EM Framework</span>

The log-likelihood can be decomposed as:

$$\log p(X|\theta) = \underbrace{\mathbb{E}_{q(Z)}[\log p(X,Z|\theta)] + H[q]}_{\text{ELBO}} + \underbrace{D_{KL}[q(Z) \| p(Z|X,\theta)]}_{\geq 0}$$

- **E-step**: Set $q(Z) = p(Z|X, \theta^{\text{old}})$ → KL = 0, ELBO = log-likelihood
- **M-step**: Maximize ELBO w.r.t. $\theta$ → guaranteed to increase log-likelihood

> **Key Insight**: EM is a special case of variational inference where the E-step uses the exact posterior over latent variables.

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">2. Setup & Imports</span>

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

tfd = tfp.distributions

tf.random.set_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print(f"TensorFlow: {tf.__version__}")
print(f"TFP: {tfp.__version__}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">3. EM for Gaussian Mixtures from Scratch</span>

In [ ]:
# Generate 2D data from a 3-component GMM
np.random.seed(42)
true_means = np.array([[-3, -2], [2, 3], [5, -1]], dtype=np.float32)
true_covs = np.array([
    [[0.8, 0.3], [0.3, 0.5]],
    [[1.0, -0.4], [-0.4, 0.8]],
    [[0.5, 0.0], [0.0, 0.7]]
], dtype=np.float32)
true_weights = np.array([0.3, 0.5, 0.2])
n_samples = 500

# Sample from true mixture
data = []
true_labels = []
for k in range(3):
    n_k = int(true_weights[k] * n_samples)
    data.append(np.random.multivariate_normal(true_means[k], true_covs[k], n_k))
    true_labels.extend([k] * n_k)
data = np.vstack(data).astype(np.float32)
true_labels = np.array(true_labels)

# Shuffle
perm = np.random.permutation(len(data))
data = data[perm]
true_labels = true_labels[perm]

print(f"Dataset: {data.shape[0]} points in {data.shape[1]} dimensions")

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.1. E-Step: Computing Responsibilities</span>

In [ ]:
def e_step(data, weights, means, covs):
    """
    E-Step: Compute responsibilities r_nk = p(z_n = k | x_n, theta)
    
    Args:
        data: (N, D) array of observations
        weights: (K,) mixing weights
        means: (K, D) component means
        covs: (K, D, D) component covariances
    
    Returns:
        responsibilities: (N, K) array
    """
    K = len(weights)
    N = len(data)
    log_resps = np.zeros((N, K))
    
    for k in range(K):
        dist = tfd.MultivariateNormalFullCovariance(
            loc=means[k], covariance_matrix=covs[k]
        )
        log_resps[:, k] = np.log(weights[k] + 1e-10) + dist.log_prob(data).numpy()
    
    # Normalize (log-sum-exp trick for numerical stability)
    log_sum = np.logaddexp.reduce(log_resps, axis=1, keepdims=True)
    log_resps -= log_sum
    
    return np.exp(log_resps)

print("✅ E-step defined: computes soft assignments p(z_n = k | x_n)")

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.2. M-Step: Updating Parameters</span>

In [ ]:
def m_step(data, responsibilities):
    """
    M-Step: Update parameters given responsibilities.
    
    Returns:
        weights, means, covs
    """
    N, K = responsibilities.shape
    D = data.shape[1]
    
    N_k = responsibilities.sum(axis=0)  # Effective number of points per cluster
    
    # Update weights
    weights = N_k / N
    
    # Update means
    means = np.zeros((K, D))
    for k in range(K):
        means[k] = (responsibilities[:, k:k+1] * data).sum(axis=0) / N_k[k]
    
    # Update covariances
    covs = np.zeros((K, D, D))
    for k in range(K):
        diff = data - means[k]  # (N, D)
        weighted_diff = responsibilities[:, k:k+1] * diff  # (N, D)
        covs[k] = (weighted_diff.T @ diff) / N_k[k]
        covs[k] += 1e-6 * np.eye(D)  # Regularization
    
    return weights.astype(np.float32), means.astype(np.float32), covs.astype(np.float32)

print("✅ M-step defined: updates weights, means, and covariances")

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.3. Full EM Algorithm</span>

In [ ]:
def compute_log_likelihood(data, weights, means, covs):
    """Compute the log-likelihood of the data under the GMM."""
    K = len(weights)
    N = len(data)
    log_probs = np.zeros((N, K))
    
    for k in range(K):
        dist = tfd.MultivariateNormalFullCovariance(
            loc=means[k], covariance_matrix=covs[k]
        )
        log_probs[:, k] = np.log(weights[k] + 1e-10) + dist.log_prob(data).numpy()
    
    return np.logaddexp.reduce(log_probs, axis=1).sum()


def em_algorithm(data, K, max_iter=100, tol=1e-6):
    """
    Full EM algorithm for Gaussian Mixture Models.
    """
    N, D = data.shape
    
    # Initialize with K-means-like initialization
    indices = np.random.choice(N, K, replace=False)
    means = data[indices].copy()
    covs = np.array([np.eye(D) for _ in range(K)], dtype=np.float32)
    weights = np.ones(K, dtype=np.float32) / K
    
    log_likelihoods = []
    history = []  # Store parameters at each step
    
    for iteration in range(max_iter):
        # E-step
        responsibilities = e_step(data, weights, means, covs)
        
        # M-step
        weights, means, covs = m_step(data, responsibilities)
        
        # Compute log-likelihood
        ll = compute_log_likelihood(data, weights, means, covs)
        log_likelihoods.append(ll)
        history.append({'weights': weights.copy(), 'means': means.copy(), 'covs': covs.copy()})
        
        if iteration > 0 and abs(ll - log_likelihoods[-2]) < tol:
            print(f"Converged at iteration {iteration}")
            break
        
        if iteration % 5 == 0:
            print(f"Iter {iteration:3d}: log-likelihood = {ll:.4f}")
    
    return weights, means, covs, responsibilities, log_likelihoods, history


# Run EM
weights, means, covs, resps, lls, history = em_algorithm(data, K=3)

print(f"\nFinal Parameters:")
for k in range(3):
    print(f"  Component {k+1}: weight={weights[k]:.3f}, mean={means[k]}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">4. Visualizing EM Convergence</span>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Log-likelihood convergence
axes[0].plot(lls, 'o-', color='b', linewidth=2, markersize=4)
axes[0].set_xlabel('EM Iteration', fontsize=14)
axes[0].set_ylabel('Log-Likelihood', fontsize=14)
axes[0].set_title('EM Convergence (Monotonically Increasing!)', fontsize=15, fontweight='bold')

# Final clustering result
hard_labels = np.argmax(resps, axis=1)
colors_map = ['r', 'b', 'g']

for k in range(3):
    mask = hard_labels == k
    axes[1].scatter(data[mask, 0], data[mask, 1], c=colors_map[k], s=15, alpha=0.5,
                    label=f'Cluster {k+1} (π={weights[k]:.2f})')

# Draw covariance ellipses
for k in range(3):
    eigenvalues, eigenvectors = np.linalg.eigh(covs[k])
    angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
    for n_std in [1, 2]:
        width = 2 * n_std * np.sqrt(eigenvalues[0])
        height = 2 * n_std * np.sqrt(eigenvalues[1])
        ell = Ellipse(means[k], width, height, angle=angle,
                      fill=False, edgecolor=colors_map[k], linewidth=2, alpha=0.7)
        axes[1].add_patch(ell)

axes[1].scatter(means[:, 0], means[:, 1], c='black', marker='X', s=200, zorder=5, edgecolors='white', linewidth=2)
axes[1].set_xlabel('$x_1$', fontsize=14)
axes[1].set_ylabel('$x_2$', fontsize=14)
axes[1].set_title('EM Result with Covariance Ellipses', fontsize=15, fontweight='bold')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">5. EM vs Gradient Descent</span>

| **Property** | **EM Algorithm** | **Gradient Descent** |
|-------------|-----------------|---------------------|
| Guaranteed monotonic | ✅ Yes (log-likelihood never decreases) | ❌ No (may oscillate) |
| Convergence | To local maximum | To local minimum of loss |
| Step size tuning | Not needed | Required (learning rate) |
| Handles constraints | Naturally (probabilities sum to 1) | Requires reparameterization |
| Generality | Specific to latent variable models | Any differentiable loss |
| TFP integration | Manual implementation | Automatic with `tf.GradientTape` |
| Speed | Often faster per iteration | May need many iterations |

> **In practice**: For GMMs, EM is the classic choice. For more complex models (deep generative models), gradient-based methods are preferred.

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">6. Connection to Variational Inference</span>

EM is actually a **special case** of variational inference (VI):

| | **EM** | **General VI** |
|---|--------|---------------|
| E-step | Exact posterior $p(Z|X,\theta)$ | Approximate posterior $q(Z)$ |
| M-step | Maximize ELBO w.r.t. $\theta$ | Maximize ELBO w.r.t. $\theta$ and $q$ |
| KL gap | Zero (exact E-step) | Non-zero (approximation) |
| Applicable when | Exact E-step is tractable | Always |

This connection is crucial for understanding:
- **Variational Autoencoders** (VAEs): Use amortized VI with neural network encoders
- **Stochastic VI**: Mini-batch version for large datasets
- **Black-box VI**: Gradient-based ELBO optimization

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">7. Key Takeaways</span>

| **Concept** | **Key Point** |
|------------|---------------|
| E-step | Compute posterior over latent variables (responsibilities) |
| M-step | Update parameters using expected sufficient statistics |
| Monotonic convergence | Log-likelihood never decreases across EM iterations |
| Local optima | EM can get stuck — use multiple random restarts |
| EM ↔ VI | EM is VI with exact E-step; foundation for modern methods |

### Next Steps
- **Notebook 02**: **Bayesian Mixture Models with MCMC** — go fully Bayesian with Dirichlet priors